# C2: NPPeco Synthesis

Synthesize the final monthly NPPeco series.
- Formula: NPPeco(t) = NPPact(t) − HANPPharv(t)
- HANPPharv(t) = cropland + forest + grassland harvest
- Input: C1 monthly NPPact + B3/B4/B5 monthly HANPPharv
- Output: monthly NPPeco (g C m⁻²) → `output/`

In [ ]:
import yaml
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
from rasterio.enums import Resampling
import rioxarray
from geocube.api.core import make_geocube
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path('../src')))
from grid_utils import create_target_grid

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
for k, v in cfg['paths'].items():
    cfg["paths"][k] = Path(v)#.expanduser().resolve()

cfg["data_root"] = Path(cfg["data_root"])

In [ ]:
da_template = create_target_grid()

# NPPeco

In [ ]:
da_NPPact    = xr.open_dataarray(cfg["paths"]["output"] / "monthly/NPPact.nc")
da_HANPPharv = xr.open_dataarray(cfg["paths"]["output"] / "monthly/HANPPharv.nc")

In [ ]:
da_NPPeco = da_NPPact - da_HANPPharv

In [ ]:
da_NPPeco\
    .to_dataset(name='NPPeco')\
    .to_netcdf(
        cfg["paths"]["output"] / 'monthly/NPPeco.nc',
        encoding={'NPPeco': {'zlib': True, 'complevel': 5}}
    )

# NPPpot

In [ ]:
da_npppot = xr.open_dataarray(cfg["paths"]["trendy"] / 'LPJ-GUESS_S2_npp.nc')\
    .sel(time=slice("2010-01-01", None))\
    .rename({"latitude": "y", "longitude": "x"})\
    .drop_attrs()\
    .rio.write_crs("EPSG:4326")\
    .rio.reproject_match(da_template, resampling=Resampling.bilinear)\
    .assign_coords(time=pd.date_range(f"2010-01-01", f"2024-12-31", freq="ME"))\
    .clip(min=0)\
    * 1000

In [ ]:
seconds_per_month = da_npppot.time.dt.days_in_month * 86400

In [ ]:
da_npppot_monthly = da_npppot * seconds_per_month

In [ ]:
da_npppot_monthly\
    .to_dataset(name='NPPpot')\
    .to_netcdf(
        cfg["paths"]["output"] / 'monthly/NPPpot.nc',
        encoding={'NPPpot': {'zlib': True, 'complevel': 5}}
    )

# HANPP

In [ ]:
da_NPPpot = xr.open_dataarray(cfg["paths"]["output"] / 'monthly/NPPpot.nc')
da_NPPeco = xr.open_dataarray(cfg["paths"]["output"] / 'monthly/NPPeco.nc')

In [ ]:
da_HANPP = da_NPPpot - da_NPPeco

In [ ]:
da_HANPP\
    .to_dataset(name='HANPP')\
    .to_netcdf(
        cfg["paths"]["output"] / 'monthly/HANPP.nc',
        encoding={'HANPP': {'zlib': True, 'complevel': 5}}
    )